## Bevezetés 

Ez a lecke a következőket fogja lefedni: 
- Mi az a függvényhívás és annak felhasználási esetei 
- Hogyan hozzunk létre függvényhívást az Azure OpenAI segítségével 
- Hogyan integráljuk a függvényhívást egy alkalmazásba 

## Tanulási célok 

A lecke elvégzése után tudni fogod, hogyan és érted: 

- A függvényhívás használatának célját 
- Függvényhívás beállítását az Azure Open AI szolgáltatás használatával 
- Hatékony függvényhívások tervezését az alkalmazásod felhasználási esetéhez 


## A függvényhívások megértése

Ehhez a leckéhez egy olyan funkciót szeretnénk készíteni oktatási startupunk számára, amely lehetővé teszi a felhasználók számára, hogy chatbot segítségével találjanak műszaki tanfolyamokat. Olyan tanfolyamokat ajánlunk, amelyek illeszkednek a készségeikhez, jelenlegi szerepükhöz és érdeklődési technológiájukhoz.

Ennek megvalósításához a következők kombinációját fogjuk használni:
 - `Azure Open AI`, hogy csevegési élményt hozzunk létre a felhasználó számára
 - `Microsoft Learn Catalog API`, hogy segítsünk a felhasználóknak tanfolyamokat találni a kérésük alapján
 - `Függvényhívás`, hogy a felhasználó lekérdezését egy függvényhez küldjük az API-kérés elvégzéséhez.

A kezdéshez nézzük meg, miért szeretnénk egyáltalán használni a függvényhívást:


### Miért a függvényhívás? 

Ha már befejeztél bármelyik másik leckét ebben a tanfolyamban, valószínűleg érted a Nagy Nyelvi Modellek (LLM-ek) használatának erejét. Remélhetőleg a korlátaikat is látod. 

A Függvényhívás az Azure Open AI Szolgáltatásának egy funkciója, amely a következő korlátok leküzdésére szolgál: 
1) Következetes válaszformátum 
2) Egy alkalmazás más forrásaiból származó adatok felhasználásának képessége csevegési kontextusban 

A függvényhívások előtt az LLM válaszai strukturálatlanok és következetlenek voltak. A fejlesztőknek bonyolult ellenőrző kódot kellett írniuk, hogy minden válaszvariációt kezelni tudjanak. 

A felhasználók nem tudtak olyan kérdésekre választ kapni, mint például: "Milyen az aktuális időjárás Stockholmban?". Ez azért volt, mert a modelleket csak a képzésük időpontjában rendelkezésre álló adatok korlátozták. 

Nézzük meg az alábbi példát, amely ezt a problémát szemlélteti: 

Tegyük fel, hogy létre akarunk hozni egy tanulói adatbázist, hogy megfelelő kurzust javasoljunk nekik. Lent két diákról szóló leírást látsz, amelyek nagyon hasonló adatokat tartalmaznak.


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


Ezt el akarjuk küldeni egy LLM-nek az adatok feldolgozásához. Ezt később alkalmazásunkban használhatjuk arra, hogy elküldjük egy API-nak vagy elmentsük egy adatbázisba.

Készítsünk két azonos promptot, amikben megadjuk az LLM-nek, hogy milyen információkra vagyunk kíváncsiak:


Ezt szeretnénk elküldeni egy LLM-nek, hogy kiszűrje azokat a részeket, amelyek fontosak a termékünk szempontjából. Így két azonos promptot hozhatunk létre az LLM utasításához: 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


Miután elkészítettük ezt a két promptot, elküldjük őket az LLM-nek a `client.responses.create` használatával. A promptot az `input` változóban tároljuk, és a szerepet `user`-re állítjuk. Ez azt hivatott utánozni, hogy egy felhasználó üzenetet ír egy chatbotnak.



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

Most mindkét kérést elküldhetjük az LLM-nek, és megvizsgálhatjuk a választ, amit kapunk. 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



Bár a promptok ugyanazok és a leírások hasonlóak, különböző formátumokat kaphatunk a `Grades` tulajdonsághoz.

Ha többször lefuttatjuk a fenti cellát, a formátum lehet `3.7` vagy `3.7 GPA`.

Ennek oka, hogy az LLM a strukturálatlan adatokat a megírt prompt formájában kapja, és szintén strukturálatlan adatokat ad vissza. Strukturált formátumra van szükségünk, hogy tudjuk, mire számítsunk az adat tárolásakor vagy használatakor.

Funkcióhívás használatával biztosíthatjuk, hogy strukturált adatokat kapjunk vissza. Funkcióhívás esetén az LLM valójában nem hív meg vagy futtat le semmilyen függvényt. Ehelyett létrehozunk egy struktúrát, amelyet az LLM követhet a válaszai során. Ezután ezeket a strukturált válaszokat használjuk arra, hogy tudjuk, melyik függvényt futtassuk az alkalmazásainkban.
 


![Függvényhívás folyamatábra](../../../../translated_images/hu/Function-Flow.083875364af4f4bb.webp)


Ezután a függvénytől kapott eredményt továbbíthatjuk az LLM-nek. Az LLM természetes nyelven fog válaszolni, hogy megválaszolja a felhasználó kérdését. 


### Használati esetek a függvényhívások alkalmazására

**Külső eszközök hívása**
A chatbotok kiválóak a felhasználói kérdésekre adott válaszok szolgáltatásában. A függvényhívások használatával a chatbotok a felhasználói üzenetek alapján bizonyos feladatokat végezhetnek el. Például egy diák megkérheti a chatbotot, hogy „Küldj emailt az oktatómnak, hogy több segítségre van szükségem ebben a tárgyban”. Ez egy `send_email(to: string, body: string)` nevű függvényhívást eredményezhet.


**API vagy adatbázis lekérdezések létrehozása**
A felhasználók természetes nyelv használatával találhatnak meg információkat, amelyek egy formázott lekérdezéssé vagy API kéréssé alakulnak. Ennek példája lehet egy tanár, aki azt kéri, hogy „Kik azok a diákok, akik teljesítették az utolsó feladatot”, ami meghívhat egy `get_completed(student_name: string, assignment: int, current_status: string)` nevű függvényt.


**Strukturált adatok létrehozása**
A felhasználók egy szövegtömböt vagy CSV fájlt használhatnak, és a nagy nyelvi modellt (LLM) arra, hogy fontos információkat nyerjenek ki belőle. Például egy diák egy Wikipédia cikket a békemegállapodásokról átalakíthat AI villanókártyák készítéséhez. Ezt egy `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` nevű függvény használatával lehet megvalósítani.


## 2. Az első függvényhívás létrehozása 

A függvényhívás létrehozásának folyamata 3 fő lépésből áll: 
1. Hívja meg a Chat Completions API-t a függvényeinek listájával és egy felhasználói üzenettel 
2. Olvassa el a modell válaszát egy művelet végrehajtásához, például egy függvény vagy API hívás végrehajtásához 
3. Tegyen egy újabb hívást a Chat Completions API-hoz a függvénye válaszával, hogy ezt az információt felhasználva válaszoljon a felhasználónak. 


![Függvényhívás folyamata](../../../../translated_images/hu/LLM-Flow.3285ed8caf4796d7.webp)


### Egy függvényhívás elemei 

#### Felhasználói bevitel 

Az első lépés egy felhasználói üzenet létrehozása. Ezt dinamikusan hozzárendelhetjük egy szövegbeviteli érték alapján, vagy itt is hozzárendelhetünk egy értéket. Ha ez az első alkalom, hogy a Chat Completions API-val dolgozol, meg kell határoznunk az üzenet `role` és `content` értékeit. 

A `role` lehet `system` (szabályok létrehozása), `assistant` (a modell) vagy `user` (a végfelhasználó). Függvényhívás esetén ezt `user`-ként állítjuk be, és egy példakérdést adunk meg. 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### Függvények létrehozása. 

Ezután definiálunk egy függvényt és annak paramétereit. Itt csak egy függvényt fogunk használni, amelynek a neve `search_courses`, de több függvényt is létrehozhatsz.

**Fontos** : A függvények beletartoznak a rendszerüzenetbe az LLM-nek, és beleszámítanak az elérhető tokenek mennyiségébe.


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**Definíciók** 

`name` - A függvény neve, amelyet meghívni szeretnénk. 

`description` - Ez a függvény működésének leírása. Itt fontos, hogy pontos és világos legyél 

`parameters` - A válaszban szerepeltetni kívánt értékek és formátum listája 


`type` - Az adattípus, amelyben a tulajdonságok tárolva lesznek 

`properties` - Az adott válaszban használni kívánt specifikus értékek listája 


`name` - a tulajdonság neve, amelyet a modell a formázott válaszában használni fog 

`type` - Ennek a tulajdonságnak az adattípusa 

`description` - A konkrét tulajdonság leírása 


**Opcionális**

`required` - a függvényhívás befejezéséhez szükséges tulajdonság 


### A függvényhívás elkészítése 
Miután definiáltuk a függvényt, most be kell illesztenünk azt a Chat Completion API hívásába. Ezt úgy tesszük meg, hogy hozzáadjuk a `functions` paramétert a kéréshez. Ebben az esetben `functions=functions`. 

Van egy lehetőség a `function_call` `auto` értékre állítására is. Ez azt jelenti, hogy hagyjuk az LLM-re, hogy dönthesse el, melyik függvényt kell meghívni a felhasználói üzenet alapján, ahelyett, hogy mi rendelnénk hozzá. 


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


Most nézzük meg a választ, és lássuk, hogyan van formázva: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

Látható, hogy a függvény neve meghívásra került, és a felhasználói üzenetből az LLM képes volt megtalálni az adatokat, amelyek illeszkednek a függvény argumentumaihoz. 


## 3. Függvényhívások integrálása egy alkalmazásba.


Miután teszteltük az LLM által formázott választ, most integrálhatjuk ezt egy alkalmazásba.

### A folyamat kezelése

Az alkalmazásba való integráláshoz tegyük meg a következő lépéseket:

Először hívjuk meg az Open AI szolgáltatásokat, és tároljuk az üzenetet egy `response_message` nevű változóban.


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


Most definiáljuk azt a függvényt, amely a Microsoft Learn API-t hívja meg a tanfolyamok listájának lekéréséhez: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



Jó gyakorlatként először megnézzük, hogy a modell szeretne-e egy függvényt hívni. Ezután létrehozzuk az elérhető függvények közül az egyiket, és összepárosítjuk azt a hívott függvénnyel. 
Ezután a függvény argumentumait hozzárendeljük az LLM-ből származó argumentumokhoz.

Végül hozzáfűzzük a függvényhívás üzenetet és a `search_courses` üzenet által visszaadott értékeket. Ez az LLM számára minden szükséges információt megad ahhoz, hogy
természetes nyelven válaszoljon a felhasználónak.


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


Most elküldjük a frissített üzenetet az LLM-nek, hogy természetes nyelvű választ kapjunk az API JSON-formátumú válasz helyett. 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    temperature=0,
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## Kód kihívás 

Nagyszerű munka! Az Azure Open AI Function Calling további elsajátításához építhetsz: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - A függvény további paraméterei, amelyek segíthetnek a tanulóknak több kurzus megtalálásában. A rendelkezésre álló API paramétereket itt találod: 
 - Hozz létre egy másik függvényhívást, amely több információt vesz a tanulótól, például az anyanyelvét 
 - Készíts hibakezelést arra az esetre, ha a függvényhívás és/vagy az API hívás nem ad vissza megfelelő kurzusokat 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
